# Clustering: Cardiovascular severity
Clustering to identify patients with different levels of cardiac impairment

Before clustering, we import the dependency libraries: 

In [1]:
!pip install pandas scikit-learn

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from collections import Counter
from pathlib import Path

And load the patient profile dataset 

In [3]:
notebook_dir = Path().resolve()
FEATURE_PATH = notebook_dir.parents[1]  / "1" / "Features" 
PATIENT_PROFILES_NAME = "patient_profiles.csv"

df = pd.read_csv(FEATURE_PATH / PATIENT_PROFILES_NAME)

## Feature selection for  cardiovascular severity clustering

We take from patient profiles features all those which rely on cardiovascular severity

In [4]:
features_severity = [
    # Synthetic scores
    'heart_failure_severity_score', 
    'cad_severity_score',
    # Cardiac structural damage
    'lv_dilated',  
    'wall_motion_abnormality', 
    'mitral_regurgitation',
    # ECG alterations
    'atrial_fibrillation', 
    'lbbb',  
    'st_depression', 
    'st_elevation',
    # Functional impairment
    'oxygen_saturation',
    # Renal function (severity marker)
    'creat_max', 
    'creat_abnormal_ratio',
    # Therapeutic intensity
    'total_procedures', 
    'procedures_days_span'
]

In [5]:
# Extract data subset
df_severity = df[['subject_id'] + features_severity].copy()

In [6]:
# Handle missing values
df_severity_clean = df_severity.dropna(subset=features_severity)
print(f"\nUsable patients: {len(df_severity_clean)} / {len(df_severity)}")


Usable patients: 4694 / 4694


In [7]:
# Prepare data for clustering
X_severity = df_severity_clean[features_severity]


## Data normalization

In [8]:
X_severity.head()

,heart_failure_severity_score,cad_severity_score,lv_dilated,wall_motion_abnormality,mitral_regurgitation,atrial_fibrillation,lbbb,st_depression,st_elevation,oxygen_saturation,creat_max,creat_abnormal_ratio,total_procedures,procedures_days_span
0,3.0,2.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,100.0,1.193922,1.0000,2.079442,0.00000
1,2.0,4.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,96.0,0.693147,0.0000,1.098612,0.00000
2,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,97.0,0.788457,0.1875,2.197225,2.70805
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,98.0,0.693147,0.0000,0.000000,0.00000
4,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,99.0,0.693147,0.0000,1.098612,0.00000


In [9]:
binary_cols = []
continuous_cols = []

for col in X_severity.columns:
    unique_vals = X_severity[col].nunique()
    if unique_vals <= 2:
        binary_cols.append(col)
    else:
        continuous_cols.append(col)

print(f"\nBinary features: {len(binary_cols)}")
print(binary_cols[:10])
print(f"\nContinuous features: {len(continuous_cols)}")
print(continuous_cols[:10])



Binary features: 7
['lv_dilated', 'wall_motion_abnormality', 'mitral_regurgitation', 'atrial_fibrillation', 'lbbb', 'st_depression', 'st_elevation']

Continuous features: 7
['heart_failure_severity_score', 'cad_severity_score', 'oxygen_saturation', 'creat_max', 'creat_abnormal_ratio', 'total_procedures', 'procedures_days_span']


We use `RobustScaler` from sklearn to normalize contnuous features (Z-score normalization)

In [10]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('binary', 'passthrough', binary_cols),      
        ('continuous', RobustScaler(), continuous_cols)  
    ]
)

X_severity_normalized = preprocessor.fit_transform(X_severity)
X_severity_normalized = pd.DataFrame(
    X_severity_normalized, 
    columns=binary_cols + continuous_cols, 
    index=X_severity.index
)

## K-means clustering 

### Optimal K derivation

In [11]:
# Range of k to test
k_range = range(2, 11)

# Evaluation metrics
inertias = []
silhouette_scores = []
calinski_scores = []
davies_bouldin_scores = []

print("\nComputing metrics for k from 2 to 10...")
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    labels = kmeans.fit_predict(X_severity_normalized)

    inertia = kmeans.inertia_
    silhouette = silhouette_score(X_severity_normalized, labels)

    inertias.append(inertia)
    silhouette_scores.append(silhouette)
    
    print(f"k={k:2d} | Inertia: {inertia:8.2f} | Silhouette: {silhouette:.4f}")


Computing metrics for k from 2 to 10...
k= 2 | Inertia: 23847.66 | Silhouette: 0.2571
k= 3 | Inertia: 20330.88 | Silhouette: 0.2392
k= 4 | Inertia: 17854.79 | Silhouette: 0.2137
k= 5 | Inertia: 16440.16 | Silhouette: 0.2175
k= 6 | Inertia: 15415.11 | Silhouette: 0.1652
k= 7 | Inertia: 14593.85 | Silhouette: 0.1713
k= 8 | Inertia: 13951.14 | Silhouette: 0.1406
k= 9 | Inertia: 13490.33 | Silhouette: 0.1433
k=10 | Inertia: 13082.03 | Silhouette: 0.1374


In [12]:
best_silhouette_k = list(k_range)[np.argmax(silhouette_scores)]
print(f" Optimal Silhouette Score: k = {best_silhouette_k}")

 Optimal Silhouette Score: k = 2


In [13]:
optimal_k = best_silhouette_k

### Training with optimal K

In [14]:
kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10, max_iter=300)
cluster_labels = kmeans_final.fit_predict(X_severity_normalized)

In [15]:
# Add labels to dataframe
df_severity_clean['cluster'] = cluster_labels

### Clusters distribution and characterization

In [16]:
cluster_counts = pd.Series(cluster_labels).value_counts().sort_index()
print("\nCluster sizes:")
for cluster_id, count in cluster_counts.items():
    percentage = (count / len(cluster_labels)) * 100
    print(f"  Cluster {cluster_id}: {count:4d} patients ({percentage:5.1f}%)")


Cluster sizes:
  Cluster 0: 1026 patients ( 21.9%)
  Cluster 1: 3668 patients ( 78.1%)


In [ ]:
df_severity_clean['cluster'] = kmeans.labels_
profile = df_severity_clean.groupby('cluster')[features_severity].mean()
print(profile.T) 

cluster                               0          1          2          3  \
heart_failure_severity_score   0.696246   0.945260   1.201313   5.027190   
cad_severity_score             0.066553   2.339119   0.096280   0.235650   
lv_dilated                     0.448805   0.634179   0.667396   0.791541   
wall_motion_abnormality        0.295222   0.515354   0.367615   0.861027   
mitral_regurgitation           0.660410   0.782377   0.715536   0.839879   
atrial_fibrillation            0.150171   0.056075   0.170678   0.184290   
lbbb                           0.023891   0.028037   0.052516   0.072508   
st_depression                  0.093857   0.137517   0.048140   0.069486   
st_elevation                   0.039249   0.066756   0.035011   0.033233   
oxygen_saturation             99.283276  97.391188  93.986871  97.148036   
creat_max                      0.702673   0.792423   0.731290   0.891067   
creat_abnormal_ratio           0.153933   0.275273   0.189649   0.414887   
total_proced

#### Centroids

In [ ]:
# Centroids in normalized space
centroids_normalized = kmeans_final.cluster_centers_

# Inverse transform for interpretability
centroids_original = scaler.inverse_transform(centroids_normalized)

# Create dataframe with centroids
centroids_df = pd.DataFrame(
    centroids_original,
    columns=features_severity,
    index=[f'Cluster {i}' for i in range(optimal_k)]
)

print("\nCentroids (original values):")
print(centroids_df.round(3))